In [27]:
import mlflow
import os
from dotenv import load_dotenv

load_dotenv()

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = 'FINAL_PROJECT'

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

if mlflow.get_experiment_by_name(name=EXPERIMENT_NAME):
    experiment_id = dict(mlflow.get_experiment_by_name(name=EXPERIMENT_NAME))['experiment_id']
    mlflow.set_experiment(experiment_id=experiment_id)
else:
    mlflow.set_experiment(EXPERIMENT_NAME)
    experiment_id = dict(mlflow.get_experiment_by_name(name=EXPERIMENT_NAME))['experiment_id']
    mlflow.set_experiment(experiment_id=experiment_id)

# ALS матрица

In [39]:
import pandas as pd

events_train = pd.read_parquet('events_train.parquet')
events_test = pd.read_parquet('events_test.parquet')

events = pd.concat([events_train, events_test])

In [40]:
#кодирование для матрицы

from sklearn.preprocessing import LabelEncoder
import joblib

user_encoder = LabelEncoder()
user_encoder.fit(events['user_id'])
events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])

item_encoder = LabelEncoder()
item_encoder.fit(events['item_id'])
events_train['item_id_enc'] = item_encoder.transform(events_train['item_id'])
events_test['item_id_enc'] = item_encoder.transform(events_test['item_id'])

In [ ]:

joblib.dump(user_encoder, 'user_encoder.pkl')
joblib.dump(item_encoder, 'item_encoder.pkl')
with mlflow.start_run(run_name='encoders_save', experiment_id=experiment_id) as run:
    mlflow.log_artifact('item_encoder.pkl')
    mlflow.log_artifact('user_encoder.pkl')

In [41]:
import scipy
import numpy as np

events_train_view_addtocart = events_train[events_train['event'].isin(['view', 'addtocart'])].copy()

weight_map = {'view': 0.3, 'addtocart': 1.0}
events_train_view_addtocart['weight'] = events_train_view_addtocart['event'].map(weight_map)
interactions_train = (events_train_view_addtocart.groupby(['user_id_enc', 'item_id_enc'])['weight'].max().reset_index())

n_users = len(user_encoder.classes_)
n_items = len(item_encoder.classes_)

user_item_matrix_train = scipy.sparse.csr_matrix(
    (interactions_train['weight'],
     (interactions_train['user_id_enc'], interactions_train['item_id_enc'])),
    shape=(n_users, n_items),
    dtype=np.float32
)

In [5]:
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(factors=50, iterations=50, regularization=0.05, random_state=42)
als_model.fit(user_item_matrix_train)

joblib.dump(als_model, 'als_model.pkl')

100%|██████████| 50/50 [02:09<00:00,  2.60s/it]


['als_model.pkl']

In [4]:
als_model = joblib.load('als_model.pkl')

/home/mle-user/mle_projects/mle-final/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# получаем список всех возможных user_id (перекодированных)
user_ids_encoded = range(len(user_encoder.classes_))

# получаем рекомендации для всех пользователей
als_recommendations = als_model.recommend(
    user_ids_encoded, 
    user_item_matrix_train[user_ids_encoded], 
    filter_already_liked_items=False, N=100)

# преобразуем полученные рекомендации в табличный формат
item_ids_enc = als_recommendations[0]
als_scores = als_recommendations[1]

als_recommendations = pd.DataFrame({
    "user_id_enc": user_ids_encoded,
    "item_id_enc": item_ids_enc.tolist(), 
    "score": als_scores.tolist()})
als_recommendations = als_recommendations.explode(["item_id_enc", "score"], ignore_index=True)

# приводим типы данных
als_recommendations["item_id_enc"] = als_recommendations["item_id_enc"].astype("int")
als_recommendations["score"] = als_recommendations["score"].astype("float")

# получаем изначальные идентификаторы
als_recommendations["user_id"] = user_encoder.inverse_transform(als_recommendations["user_id_enc"])
als_recommendations["item_id"] = item_encoder.inverse_transform(als_recommendations["item_id_enc"])
als_recommendations = als_recommendations.drop(columns=["user_id_enc", "item_id_enc"])

als_recommendations.to_parquet('als_recommendations.parquet')

# Контентная матрица (не используется)

In [10]:
import pandas as pd

item_categories = pd.read_parquet('item_categories.parquet')

In [16]:
#кодирование itemid для матрицы

from sklearn.preprocessing import LabelEncoder
import joblib

cat_item_enc = LabelEncoder()
item_categories['item_id_enc'] = cat_item_enc.fit_transform(item_categories['item_id'])
joblib.dump(cat_item_enc, 'cat_item_encoder.pkl')

['cat_item_encoder.pkl']

In [ ]:
#кодивование category для матрицы

cat_enc = LabelEncoder()
item_categories['category_enc'] = cat_enc.fit_transform(item_categories['category'])
joblib.dump(cat_enc, 'cat_encoder.pkl')

['cat_encoder.pkl']

In [18]:
import scipy
import numpy as np

item_enc = LabelEncoder()
item_categories['item_id_enc'] = item_enc.fit_transform(item_categories['item_id'])
cat_enc = LabelEncoder()
item_categories['category_enc'] = cat_enc.fit_transform(item_categories['category'])

n_items = item_categories['item_id_enc'].nunique()
n_categories = item_categories['category_enc'].nunique()

category_csr = scipy.sparse.csr_matrix(
    (np.ones(len(item_categories)),
    (item_categories['item_id_enc'], item_categories['category_enc'])),
    shape=(n_items, n_categories),
    dtype=np.int8)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances

user_ids_encoded = range(len(user_encoder.classes_))
results = []

for user in user_ids_encoded: 
    user_id = user
    user_events = interactions_train.query("user_id_enc == @user_id")[["item_id_enc", "weight"]]
    if len(user_events) == 0:
        continue 
    user_items_categories_csr = category_csr[user_events["item_id_enc"].values]

    user_weights = user_events["weight"].to_numpy()
    user_weights = np.expand_dims(user_weights, axis=1)

    user_items_categories_weighted = user_items_categories_csr.multiply(user_weights)

    user_category_scores = np.asarray(user_items_categories_weighted.mean(axis=0))

    # вычисляем сходство между вектором пользователя и векторами по книгам
    similarity_scores = euclidean_distances(category_csr, user_category_scores)

    # преобразуем в одномерный массив
    similarity_scores = similarity_scores.flatten()

    if len(user_events) <= 2:
        k = 5
    else:
        k = 100

    top_k_indices = np.argsort(similarity_scores)[:k]
    top_k_scores = similarity_scores[top_k_indices]

    for item_id, score in zip(top_k_indices, top_k_scores):
        results.append({'user_id': user_id, 'item_id': item_id, 'cnt_score': score})
        
content_recommendations = pd.DataFrame(results)
content_recommendations.to_parquet('content_recommendations.parquet')


# Построение дополнительных признаков

признаки:
1. общее кол-во интеракций у юзера
2. общее кол-во интеракций у айтема
3. общее кол-во просмотров у юзера
4. общее кол-во покупок у юзера
5. общее кол-во просмотров у айтема
6. общее кол-во покупок у айтема
7. категория айтема

In [ ]:
import pandas as pd

items = pd.read_parquet('item_categories.parquet')
events_train = pd.read_parquet('events_train.parquet')
events_test = pd.read_parquet('events_test.parquet')

category_tree = pd.read_csv('category_tree.csv')
category_tree = category_tree.dropna()
category_tree['parentid'] = category_tree['parentid'].astype('int32')

In [5]:
# общая категория

category_tree_dict = dict(zip(category_tree['categoryid'], category_tree['parentid']))

def get_root_category(cat_id, tree_dict):
    while cat_id in tree_dict:
        cat_id = tree_dict[cat_id]
    return int(cat_id)

items['general_category'] = items['category'].astype('int64').apply(lambda c: get_root_category(c, category_tree_dict))

In [6]:
# просмотры пары юзер-айтем
user_item_views_train = events_train[events_train['event'] == 'view'].groupby(['user_id', 'item_id']).size().reset_index(name='user_item_views_count')
user_item_views_test = events_test[events_test['event'] == 'view'].groupby(['user_id', 'item_id']).size().reset_index(name='user_item_views_count')

# просмотры
user_total_views_train = events_train[events_train['event'] == 'view'].groupby('user_id').size().reset_index(name='user_views_count')
user_total_views_test = events_test[events_test['event'] == 'view'].groupby('user_id').size().reset_index(name='user_views_count')

item_total_views_train = events_train[events_train['event'] == 'view'].groupby('item_id').size().reset_index(name='item_views_count')
item_total_views_test = events_test[events_test['event'] == 'view'].groupby('item_id').size().reset_index(name='item_views_count')

# добавления в корзину
user_total_addtocart_train = events_train[events_train['event'] == 'addtocart'].groupby('user_id').size().reset_index(name='user_addtocart_count')
user_total_addtocart_test = events_test[events_test['event'] == 'addtocart'].groupby('user_id').size().reset_index(name='user_addtocart_count')

item_total_addtocart_train = events_train[events_train['event'] == 'addtocart'].groupby('item_id').size().reset_index(name='item_addtocart_count')
item_total_addtocart_test = events_test[events_test['event'] == 'addtocart'].groupby('item_id').size().reset_index(name='item_addtocart_count')

# все события
user_total_events_train = events_train.groupby('user_id').size().reset_index(name='user_events_count')
user_total_events_test = events_test.groupby('user_id').size().reset_index(name='user_events_count')

item_total_events_train = events_train.groupby('item_id').size().reset_index(name='item_events_count')
item_total_events_test = events_test.groupby('item_id').size().reset_index(name='item_events_count')

In [ ]:
features_train = (
    user_item_views_train
    

,user_id,user_addtocart_count
0,150,1
1,299,1
2,302,1
3,318,1
4,363,1
...,...,...
25521,1407355,1
25522,1407398,1
25523,1407437,1
25524,1407512,8


# Двухстадийный подход

In [4]:
import pandas as pd

events_train = pd.read_parquet('events_train.parquet')
events_test = pd.read_parquet('events_test.parquet')
als_recommendations = pd.read_parquet('als_recommendations.parquet')

In [5]:
# добавляем таргет к кандидатам со значением:
# — 1 для тех item_id, которые пользователь прочитал
# — 0, для всех остальных 

events_train_addtocart = events_train[events_train['event'] == 'addtocart'][['user_id', 'item_id']].copy()
events_train_addtocart['target'] = 1
candidates = als_recommendations.merge(events_train_addtocart[["user_id", "item_id", "target"]], 
                              on=['user_id', 'item_id'],
                              how='left')
candidates["target"] = candidates["target"].fillna(0).astype("int")

# в кандидатах оставляем только тех пользователей, у которых есть хотя бы один положительный таргет
candidates_to_sample = candidates.groupby("user_id").filter(lambda x: x["target"].sum() > 0)

# для каждого пользователя оставляем только 4 негативных примера
negatives_per_user = 4
candidates_for_train = pd.concat([
    candidates_to_sample[candidates_to_sample['target'] == 1],
    candidates_to_sample.query("target == 0") \
        .groupby("user_id") \
        .apply(lambda x: x.sample(min(len(x), negatives_per_user), random_state=42))
    ])

candidates_to_rank = als_recommendations[als_recommendations["user_id"].isin(events_test["user_id"].drop_duplicates())]

/tmp/ipykernel_2797/632217300.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), negatives_per_user), random_state=42))


In [6]:
# совмещение всех признаков для обучения

candidates_for_train = (
    candidates_for_train
    .merge(user_item_views_train, on=['user_id', 'item_id'], how='left')
    .merge(user_total_views_train, on='user_id', how='left')
    .merge(user_total_addtocart_train, on='user_id', how='left')
    .merge(item_total_views_train, on='item_id', how='left')
    .merge(item_total_addtocart_train, on='item_id', how='left')
    .merge(user_total_events_train, on='user_id', how='left')
    .merge(item_total_events_train, on='item_id', how='left')
    .merge(items, on='item_id', how='left')
)

candidates_to_rank = (
    candidates_to_rank
    .merge(user_item_views_test, on=['user_id', 'item_id'], how='left')
    .merge(user_total_views_test, on='user_id', how='left')
    .merge(user_total_addtocart_test, on='user_id', how='left')
    .merge(item_total_views_test, on='item_id', how='left')
    .merge(item_total_addtocart_test, on='item_id', how='left')
    .merge(user_total_events_test, on='user_id', how='left')
    .merge(item_total_events_test, on='item_id', how='left')
    .merge(items, on='item_id', how='left')
)

In [7]:
# предобработка данных перед обучением

cols_to_fix = ['user_item_views_count', 'user_views_count', 'item_views_count', 'user_addtocart_count', 
               'item_addtocart_count', 'user_events_count', 'item_events_count', 'general_category']

candidates_for_train[cols_to_fix] = candidates_for_train[cols_to_fix].fillna(0).astype('int32')
candidates_for_train['target'] = candidates_for_train['target'].astype('int8')

candidates_for_train = candidates_for_train.fillna(0)
candidates_to_rank = candidates_to_rank.fillna(0)

candidates_to_rank[cols_to_fix] = candidates_to_rank[cols_to_fix].astype('int32')

candidates_for_train = candidates_for_train.rename(columns={'score': 'als_score'})
candidates_to_rank = candidates_to_rank.rename(columns={'score': 'als_score'})

In [8]:
from catboost import CatBoostClassifier, Pool

# задаём имена колонок признаков и таргета
features = ['als_score', 'user_item_views_count', 'user_views_count', 'item_views_count', 'user_addtocart_count',
             'item_addtocart_count', 'user_events_count', 'item_events_count', 'category', 'general_category']
cat_features = ['category', 'general_category']
target = 'target'

# создаём Pool
train_data = Pool(
    data=candidates_for_train[features],
    cat_features=cat_features,
    label=candidates_for_train[target])

# инициализируем модель CatBoostClassifier
cb_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    verbose=100,
    random_seed=0,
)

# тренируем модель
cb_model.fit(train_data)

0:	learn: 0.5401587	total: 95.7ms	remaining: 1m 35s
100:	learn: 0.0912457	total: 3.75s	remaining: 33.4s
200:	learn: 0.0841250	total: 7.37s	remaining: 29.3s
300:	learn: 0.0788543	total: 11s	remaining: 25.6s
400:	learn: 0.0745810	total: 14.6s	remaining: 21.8s
500:	learn: 0.0711321	total: 18.3s	remaining: 18.2s
600:	learn: 0.0684549	total: 21.9s	remaining: 14.5s
700:	learn: 0.0657195	total: 25.5s	remaining: 10.9s
800:	learn: 0.0633515	total: 29.2s	remaining: 7.26s
900:	learn: 0.0613273	total: 32.9s	remaining: 3.62s
999:	learn: 0.0594133	total: 36.5s	remaining: 0us


CatBoostClassifier(depth=6, iterations=1000, learning_rate=0.1, loss_function='Logloss', random_seed=0, verbose=100)

In [9]:
inference_data = Pool(
    data=candidates_to_rank[features],
    cat_features=cat_features
    )
predictions = cb_model.predict_proba(inference_data)

In [10]:
candidates_to_rank["cb_score"] = predictions[:, 1]

# для каждого пользователя проставим rank, начиная с 1 — это максимальный cb_score
candidates_to_rank = candidates_to_rank.sort_values(["user_id", "cb_score"], ascending=[True, False])
candidates_to_rank["rank"] = candidates_to_rank.groupby("user_id").cumcount() + 1

max_recommendations_per_user = 100
final_recommendations = candidates_to_rank.query("rank <= @max_recommendations_per_user")

In [13]:
final_recommendations

,als_score,user_id,item_id,user_item_views_count,user_views_count,user_addtocart_count,item_views_count,item_addtocart_count,user_events_count,item_events_count,category,general_category,cb_score,rank
29,0.0,1,147,0,1,0,19,4,1,26,646,1600,0.009691,1
92,0.0,1,19,0,1,0,10,1,1,12,1171,1532,0.009031,2
81,0.0,1,42,0,1,0,14,2,1,17,84,140,0.003409,3
7,0.0,1,199,0,1,0,16,1,1,17,1163,395,0.001173,4
89,0.0,1,25,0,1,0,9,2,1,12,72,140,0.000839,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31112793,0.0,1407574,17,0,1,0,0,0,1,0,1265,140,0.000003,96
31112710,0.0,1407574,190,0,1,0,3,0,1,3,1255,791,0.000003,97
31112766,0.0,1407574,71,0,1,0,0,0,1,0,977,1224,0.000003,98
31112706,0.0,1407574,204,0,1,0,0,0,1,0,1250,791,0.000002,99


In [14]:
final_recommendations['category'] = final_recommendations['category'].astype('int32')
final_recommendations.to_parquet('recommendations.parquet')

# Метрики

In [ ]:
import pandas as pd

events_train = pd.read_parquet('events_train.parquet')
events_test = pd.read_parquet('events_test.parquet')
final_recommendations = pd.read_parquet('recommendations.parquet')
als_recommendations = pd.read_parquet('als_recommendations.parquet')

In [15]:
# NDCG@10

import numpy as np

def dcg_at_k(ranked_items, relevant_items, k):
    ranked_items = ranked_items[:k]
    dcg = 0.0
    for i, item in enumerate(ranked_items, start=1):
        rel = 1 if item in relevant_items else 0
        dcg += rel / np.log2(i + 1)
    return dcg

def ndcg_at_k(ranked_items, relevant_items, k):
    ideal_dcg = dcg_at_k(list(relevant_items), relevant_items, k)  # best possible ordering
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(ranked_items, relevant_items, k) / ideal_dcg

def mean_ndcg_at_k(user_rankings, user_relevant, k):
    scores = []
    for user_id, ranked_items in user_rankings.items():
        relevant_items = user_relevant.get(user_id, set())
        scores.append(ndcg_at_k(ranked_items, relevant_items, k))
    return np.mean(scores)

In [20]:
# топ популярных
top_k_pop_items = events_train.groupby('item_id').size().reset_index(name='count').sort_values(['count'], ascending=False)[:10]

users_train = events_train["user_id"].drop_duplicates()
users_test = events_test["user_id"].drop_duplicates()

cold_users = set(users_test) - set(users_train)

cold_users_events_with_recs = events_test[events_test["user_id"].isin(cold_users)].copy()
cold_users_events_with_recs["target"] = cold_users_events_with_recs["item_id"].isin(top_k_pop_items["item_id"]).astype(int)

precision = cold_users_events_with_recs.groupby("user_id")["target"].sum() / 100
precision_top = precision.mean()
print(f'TOP Precision: {precision_top:.6f}')

recall_top = cold_users_events_with_recs.groupby("user_id")["target"].mean().mean()
print(f'TOP Recall: {recall_top:.6f}')

cold_user_rankings = {user_id: top_k_pop_items['item_id'].tolist() for user_id in cold_users}
cold_user_relevant = (events_test[events_test['user_id'].isin(cold_users)]
                        .groupby('user_id')['item_id']
                        .apply(set)
                        .to_dict())

ndcg_pop = mean_ndcg_at_k(cold_user_rankings, cold_user_relevant, k=10)
print(f'TOP NDCG: {ndcg_pop:.6f}')

TOP Precision: 0.000084
TOP Recall: 0.004754
TOP NDCG: 0.002347


In [21]:
# персональные ALS

def process_events_recs_for_binary_metrics(events_train, events_test, recs, top_k=None):

    """
    размечает пары <user_id, item_id> для общего множества пользователей признаками
    - gt (ground truth)
    - pr (prediction)
    top_k: расчёт ведётся только для top k-рекомендаций
    """

    events_test["gt"] = True
    common_users = set(events_test["user_id"]) & set(recs["user_id"])

    print(f"Common users: {len(common_users)}")
    
    events_for_common_users = events_test[events_test["user_id"].isin(common_users)].copy()
    recs_for_common_users = recs[recs["user_id"].isin(common_users)].copy()

    recs_for_common_users = recs_for_common_users.sort_values(["user_id", "score"], ascending=[True, False])

    # оставляет только те item_id, которые были в events_train, 
    # т. к. модель не имела никакой возможности давать рекомендации для новых айтемов
    events_for_common_users = events_for_common_users[events_for_common_users["item_id"].isin(events_train["item_id"].unique())]

    if top_k is not None:
        recs_for_common_users = recs_for_common_users.groupby("user_id").head(top_k)
    
    events_recs_common = events_for_common_users[["user_id", "item_id", "gt"]].merge(
        recs_for_common_users[["user_id", "item_id", "score"]], 
        on=["user_id", "item_id"], how="outer")    

    events_recs_common["gt"] = events_recs_common["gt"].fillna(False)
    events_recs_common["pr"] = ~events_recs_common["score"].isnull()
    
    events_recs_common["tp"] = events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fp"] = ~events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fn"] = events_recs_common["gt"] & ~events_recs_common["pr"]

    return events_recs_common, recs_for_common_users, events_for_common_users

In [17]:
def compute_cls_metrics(events_recs_for_binary_metrics):
    
    groupper = events_recs_for_binary_metrics.groupby("user_id")

    # precision = tp / (tp + fp)
    precision = groupper["tp"].sum()/(groupper["tp"].sum()+groupper["fp"].sum())
    precision = precision.fillna(0).mean()
    
    # recall = tp / (tp + fn)
    recall = groupper["tp"].sum()/(groupper["tp"].sum()+groupper["fn"].sum())
    recall = recall.fillna(0).mean()

    return precision, recall

In [23]:
# als metrics

events_recs_for_binary_metrics, recs_for_common_users, events_for_common_users = process_events_recs_for_binary_metrics(
  events_train,
    events_test, 
    als_recommendations, 
    top_k=10)

precision_als, recall_als = compute_cls_metrics(events_recs_for_binary_metrics)

user_rankings = (recs_for_common_users.groupby('user_id')['item_id']
                                        .apply(list)
                                        .to_dict())

user_relevant = (events_for_common_users.groupby('user_id')['item_id']
                                          .apply(set)
                                          .to_dict())

ndcg_als = mean_ndcg_at_k(user_rankings, user_relevant, k=10)

print(f'Precision: {precision_als:.6f}')
print(f'Recall: {recall_als:.6f}')
print(f'NDCG: {ndcg_als:.6f}')

Common users: 311128


/tmp/ipykernel_2797/332359964.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  events_recs_common["gt"] = events_recs_common["gt"].fillna(False)


Precision: 0.000602
Recall: 0.002361
NDCG: 0.001765


In [19]:
# итоговые

# для экономии ресурсов оставим события только тех пользователей, 
# для которых следует оценить рекомендации
events_inference = pd.concat([events_train, events_test])
events_inference = events_inference[events_inference["user_id"].isin(events_test["user_id"].drop_duplicates())]

cb_events_recs_for_binary_metrics_5, recs_for_common_users, events_for_common_users = process_events_recs_for_binary_metrics(
    events_inference,
    events_test,
    final_recommendations.rename(columns={"cb_score": "score"}), 
    top_k=10)

cb_precision_10, cb_recall_10 = compute_cls_metrics(cb_events_recs_for_binary_metrics_5)

user_rankings = (recs_for_common_users.groupby('user_id')['item_id']
                                        .apply(list)
                                        .to_dict())

user_relevant = (events_for_common_users.groupby('user_id')['item_id']
                                          .apply(set)
                                          .to_dict())

ndcg_cb = mean_ndcg_at_k(user_rankings, user_relevant, k=10)

print(f'Precision: {cb_precision_10:.6f}')
print(f'Recall: {cb_recall_10:.6f}')
print(f'NDCG: {ndcg_cb:.6f}')

Common users: 311128


/tmp/ipykernel_2797/332359964.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  events_recs_common["gt"] = events_recs_common["gt"].fillna(False)


Precision: 0.001084
Recall: 0.004195
NDCG: 0.004433


In [22]:
cb_model.get_feature_importance(prettified=True)

,Feature Id,Importances
0,user_item_views_count,50.289879
1,item_addtocart_count,11.086434
2,als_score,10.164983
3,user_views_count,7.342911
4,category,4.865260
5,user_addtocart_count,3.988837
6,item_views_count,3.640523
7,item_events_count,3.588472
8,general_category,2.576341
9,user_events_count,2.456361


In [24]:
import mlflow
import os
from dotenv import load_dotenv
from mlflow.models import infer_signature

load_dotenv()

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = 'FINAL_PROJECT'

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

if mlflow.get_experiment_by_name(name=EXPERIMENT_NAME):
    experiment_id = dict(mlflow.get_experiment_by_name(name=EXPERIMENT_NAME))['experiment_id']
    mlflow.set_experiment(experiment_id=experiment_id)
else:
    mlflow.set_experiment(EXPERIMENT_NAME)
    experiment_id = dict(mlflow.get_experiment_by_name(name=EXPERIMENT_NAME))['experiment_id']
    mlflow.set_experiment(experiment_id=experiment_id)

#signature = infer_signature(candidates_for_train.drop(columns=['target']).head(5), final_recommendations.head(5))
#input_example = candidates_to_rank.head(5)
pip_requirements = 'requirements.txt'
params = cb_model.get_params()

metrics = {
    'top10_precision': precision_top,
    'top10_recall': recall_top,
    'top10_NDCG': ndcg_pop,
    'als_precision': precision_als,
    'als_recall': recall_als,
    'cb_precision': cb_precision_10,
    'cb_recall': cb_recall_10,
    'cb_ndcg': ndcg_cb
}

with mlflow.start_run(run_name='cb_version_4_gen_cat_new_metric_methodology', experiment_id=experiment_id) as run:

    mlflow.log_metrics(metrics)
    mlflow.log_params(params)
    mlflow.catboost.log_model(
        cb_model=cb_model,
        artifact_path='models',
        #signature=signature,
        registered_model_name='cb_ranking_model',
        pip_requirements=pip_requirements,
        #input_example=input_example
    )

/home/mle-user/mle_projects/mle-final/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/08/27 13:16:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'cb_ranking_model' already exists. Creating a new version of this model...
2026/08/27 13:16:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: cb_ranking_model, version 6
Created version '6' of model 'cb_ranking_model'.


🏃 View run cb_version_4_gen_cat_new_metric_methodology at: http://127.0.0.1:5000/#/experiments/12/runs/fb2272d6ba7649999b1dc3f8cdd7fc11
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/12
